# Index-Space Normals & Permutation Matrix Tutorial

This notebook demonstrates:
1. Computing index-space (topological) normals for block faces
2. Computing the 3x3 permutation matrix from diagonal corner pairing
3. Validating that a face match produces a correct permutation matrix
4. Visualizing normals with 3D quiver plots

In [ ]:
import numpy as np
from plot3d.normals import (
    index_space_normal,
    compute_permutation_matrix,
    validate_connectivity,
    compute_all_normals,
    export_normals_json,
    import_normals_json,
    plot_face_normals,
)

## 1. Index-Space Normals

An index-space normal is a unit integer vector with exactly one non-zero component:
- `-1` on a **low** face (constant index == 0)
- `+1` on a **high** face (constant index > 0)

Six faces per block: `imin(-1,0,0)`, `imax(+1,0,0)`, `jmin(0,-1,0)`, `jmax(0,+1,0)`, `kmin(0,0,-1)`, `kmax(0,0,+1)`

In [ ]:
# K-face at k=0 (kmin) -> normal = (0, 0, -1)
n = index_space_normal([0, 0, 0], [24, 408, 0])
print(f"kmin normal: {n}")

# J-face at j=0 (jmin) -> normal = (0, -1, 0)
n = index_space_normal([0, 0, 0], [408, 0, 24])
print(f"jmin normal: {n}")

# I-face at i=24 (imax) -> normal = (+1, 0, 0)
n = index_space_normal([24, 0, 0], [24, 408, 12])
print(f"imax normal: {n}")

## 2. Permutation Matrix from Diagonal Corners

The 3x3 integer permutation matrix maps index coordinates from one block
face to the adjacent block face at an interface:

```
j = a2 + M * (i - a1)
```

where `M` is the 3x3 matrix, `a1/a2` are the reference (lb) corners,
and `a1<->a2`, `b1<->b2` must be spatially coincident.

In [ ]:
# In-plane example: K-face <-> K-face, same orientation
# Block 1: K=0 face, a1=(0,0,0), b1=(24,408,0)
# Block 2: K=12 face, a2=(0,0,12), b2=(24,408,12)
M_in_plane = compute_permutation_matrix(
    [0, 0, 0], [24, 408, 0],   # block 1 diagonal
    [0, 0, 12], [24, 408, 12],  # block 2 diagonal
)
print("In-plane (K<->K) permutation matrix:")
print(M_in_plane)
print(f"Expected: [[1,0,0],[0,1,0],[0,0,1]]  (identity -- same orientation)")
print(f"det = {int(np.round(np.linalg.det(M_in_plane.astype(float))))}")

In [ ]:
# Cross-plane example: K-face <-> J-face
# Block 1: K=0 face, a1=(0,0,0), b1=(24,408,0)
# Block 2: J=0 face, a2=(408,0,0), b2=(0,0,24) -- DIRECTED (not ascending)
M_cross = compute_permutation_matrix(
    [0, 0, 0], [24, 408, 0],    # block 1 diagonal
    [408, 0, 0], [0, 0, 24],    # block 2 diagonal (directed!)
)
print("Cross-plane (K<->J) permutation matrix:")
print(M_cross)
print(f"Expected: [[0,-1,0],[0,0,-1],[1,0,0]]")
print(f"det = {int(np.round(np.linalg.det(M_cross.astype(float))))}")

In [ ]:
# BAD pairing: ascending bounds for block2 (not directed)
# This should produce None or an incorrect matrix
M_bad = compute_permutation_matrix(
    [0, 0, 0], [24, 408, 0],
    [0, 0, 0], [408, 0, 24],  # ascending -- WRONG
)
if M_bad is None:
    print("Bad pairing correctly returned None")
else:
    print(f"Bad pairing produced: {M_bad.tolist()}")
    print("(This is different from the correct cross-plane matrix above)")

## 3. Batch Validation

Validate a list of face matches to ensure all produce valid permutation matrices.

In [ ]:
# Example face matches (as they appear in connectivity.json)
face_matches = [
    {
        "block1": {"block_index": 0, "lb": [0, 0, 0], "ub": [24, 408, 0]},
        "block2": {"block_index": 1, "lb": [0, 0, 12], "ub": [24, 408, 12]},
    },
    {
        "block1": {"block_index": 0, "lb": [0, 0, 0], "ub": [24, 408, 0]},
        "block2": {"block_index": 421, "lb": [408, 0, 0], "ub": [0, 0, 24]},
    },
]

results = validate_connectivity(face_matches)
for r in results:
    status = "VALID" if r["valid"] else f"INVALID: {r['error']}"
    print(f"Match {r['match_index']}: {status}")
    if r["matrix"]:
        print(f"  Matrix: {r['matrix']}")

## 4. Normals JSON Export/Import

Compute normals for all 6 faces of each block and save to `normals.json`.

In [ ]:
# To use with real blocks, load a Plot3D mesh:
# from plot3d import read_plot3D
# blocks = read_plot3D('mesh.p3d', binary=False)
# normals = compute_all_normals(blocks)
# export_normals_json(normals, 'normals.json')
#
# To re-load:
# normals = import_normals_json('normals.json')

print("See grid-packed/scripts/normals_example.ipynb for a worked example with real mesh data.")

## 5. Visualization

Use `plot_face_normals()` to draw 3D quiver arrows at face centroids.

Color coding:
- **Red** shades: I-faces (imin/imax)
- **Green** shades: J-faces (jmin/jmax)
- **Blue** shades: K-faces (kmin/kmax)

In [ ]:
# To visualize with real blocks:
# from plot3d import read_plot3D
# blocks = read_plot3D('mesh.p3d', binary=False)
# ax = plot_face_normals(blocks)

print("Load a Plot3D mesh and call plot_face_normals(blocks) to visualize.")
print("See grid-packed/scripts/normals_example.ipynb for a live example.")